# Summarization Evaluation: Fine-tuned vs. Pretrained t5-small

Compares the t5-small model fine-tuned on real MTSamples clinical transcriptions
against the plain pretrained t5-small baseline, on two real held-out datasets:

1. **MTSamples test split** (in-domain): real clinical transcription -> real description
2. **PubMed sample** (out-of-domain): real research article -> real abstract

This checks both whether fine-tuning improved in-domain summaries, and whether it
hurt the model's general summarization ability outside the clinical-note domain.

In [1]:
import os
import sys
from pathlib import Path

# Run from the repo root regardless of where this notebook was launched from,
# since src/ modules use paths (e.g. models/summarizer/final) relative to it.
if not Path('pytest.ini').exists() and Path('../pytest.ini').exists():
    os.chdir('..')
sys.path.append(os.getcwd())

from src.modeling.evaluate_summarizer import (
    evaluate_on_mtsamples,
    evaluate_on_pubmed,
    FINE_TUNED_DIR,
)
from src.modeling.train_summarizer import MODEL_NAME
import pandas as pd

In [2]:
assert FINE_TUNED_DIR.exists(), "Run `python -m src.modeling.train_summarizer` first"

rows = []
for label, path in [
    ("pretrained-t5-small (baseline)", MODEL_NAME),
    ("fine-tuned-t5-small (ours)", str(FINE_TUNED_DIR)),
]:
    mt = evaluate_on_mtsamples(path)
    pm = evaluate_on_pubmed(path)
    rows.append({"model": label, "dataset": "MTSamples (in-domain)", **mt})
    rows.append({"model": label, "dataset": "PubMed (out-of-domain)", **pm})

results = pd.DataFrame(rows)
results

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

,model,dataset,rouge1,rouge2,rougeL,rougeLsum
0,pretrained-t5-small (baseline),MTSamples (in-domain),0.2306,0.1326,0.2041,0.2035
1,pretrained-t5-small (baseline),PubMed (out-of-domain),0.1756,0.0586,0.1240,0.1520
2,fine-tuned-t5-small (ours),MTSamples (in-domain),0.6298,0.5654,0.6193,0.6158
3,fine-tuned-t5-small (ours),PubMed (out-of-domain),0.1400,0.0512,0.1062,0.1249


## Example outputs side by side

A few real MTSamples transcriptions with the human-written reference description,
the pretrained baseline's summary, and the fine-tuned model's summary.

In [3]:
from src.data_ingestion.load_mtsamples import load_mtsamples
from src.modeling.evaluate_summarizer import summarize_batch

sample = load_mtsamples().sample(5, random_state=123)

baseline_preds = summarize_batch(MODEL_NAME, sample["transcription"].tolist())
finetuned_preds = summarize_batch(str(FINE_TUNED_DIR), sample["transcription"].tolist())

comparison = pd.DataFrame({
    "reference_description": sample["description"].values,
    "pretrained_baseline": baseline_preds,
    "fine_tuned": finetuned_preds,
})
pd.set_option("display.max_colwidth", 120)
comparison

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

,reference_description,pretrained_baseline,fine_tuned
0,"The patient is a 26-year-old gravida 2, para 1-0-0-1, at 28-1/7 weeks who presents to the emergency room with left l...","gravida 2, para 1-0-0-1, 26, presents to emergency room with left lower quadrant pain. she reports no bowel movement...","The patient presents to the emergency room with left lower quadrant pain, reports no bowel movement in two weeks as ..."
1,Headache. Right frontal lobe glioma.,a patient with a occipital headaches was diagnosed with a xanthomatous astrocytoma. she underwent a right frontal ma...,MRI Brain: Large solid and cystic right frontal lobe mass with a large amount of surrounding edema.
2,Left L4-L5 transforaminal neuroplasty with nerve root decompression and lysis of adhesions followed by epidural ster...,"375 units of Wydase were injected through each needle. after two minutes, 3.5 cc of 0.5% Marcaine and 80 mg of Depo-...",Left L4-L5 transforaminal neuroplasty with nerve root decompression and lysis of adhesions followed by epidural ster...
3,"Ethmoidectomy, antrostomy with polyp removal, turbinectomy, and septoplasty.",the patient is a 55-year-old female who has had chronic nasal obstruction secondary to nasal polyps and chronic sinu...,"Bilateral endoscopic sinus surgery, including left anterior and posterior ethmoidectomy, left maxillary antrostomy w..."
4,OssaTron extracorporeal shockwave therapy to right lateral epicondyle. Right lateral epicondylitis.,the patient was positioned for extracorporeal shockwave therapy. the nose piece was placed against the lateral epico...,OssaTron extracorporeal shockwave therapy to right lateral epicondyle.
